## Rule-based data integration pipeline (Standard Blocker)

In [1]:
from pathlib import Path

ROOT = Path.cwd()

DATA_DIR = ROOT / "parquet"
OUTPUT_DIR = ROOT / "output"
MLDS_DIR = ROOT / "ml-datasets"
BLOCK_EVAL_DIR = OUTPUT_DIR / "blocking_evaluation"
CORR_DIR = OUTPUT_DIR / "correspondences"

PIPELINE_DIR = OUTPUT_DIR / "data_fusion"
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
from PyDI.io import load_parquet

amazon_books = load_parquet(
    DATA_DIR / "amazon_new.parquet",
    name="amazon_books"
)

goodreads = load_parquet(
    DATA_DIR / "goodreads_new.parquet",
    name="goodreads"
)

metabooks = load_parquet(
  DATA_DIR / "metabooks_new.parquet",
  name="metabooks"
)

In [3]:
from PyDI.entitymatching import StandardBlocker
from PyDI.entitymatching import StringComparator, NumericComparator
from PyDI.entitymatching import RuleBasedMatcher


# standard blocker: metabooks > amazon
st_blocker_m2a = StandardBlocker(
    metabooks, amazon_books,
    on=['title_norm','author_norm'],
    batch_size=1000,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

# standard blocker: metabooks > goodreads
st_blocker_m2g = StandardBlocker(
    metabooks, goodreads,
    on=['title_norm','author_norm'],
    batch_size=1000,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

/Users/abd/Developer/team_project_data_integration/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
comparators_m2a = [
    StringComparator(column="title_norm", similarity_function="cosine"),
    StringComparator(column="author_norm", similarity_function="cosine", preprocess=str.lower),
    StringComparator(column="publisher", similarity_function="cosine", preprocess=str.lower),
    StringComparator(column="isbn_clean", similarity_function="cosine"),
    NumericComparator(column="publish_year", max_difference=1),
]

comparators_m2g = comparators_m2a + [
    StringComparator(column="genres", similarity_function="jaccard",
                     preprocess=str.lower, list_strategy="concatenate"),
    NumericComparator(column="price", max_difference=5),
    NumericComparator(column="page_count", max_difference=10),
    NumericComparator(column="rating", max_difference=0.2),
]

In [14]:
matcher = RuleBasedMatcher()

# matching metabooks > amazon
st_corr_m2a = matcher.match(
    df_left=metabooks,
    df_right=amazon_books, 
    candidates=st_blocker_m2a,
    comparators=comparators_m2a,
    weights=[0.4, 0.2, 0.1, 0.2, 0.1], 
    threshold=0.7,
    id_column='id'
)

# matching metabooks > goodreads
st_corr_m2g = matcher.match(
    df_left=metabooks,
    df_right=goodreads, 
    candidates=st_blocker_m2g,
    comparators=comparators_m2g,
    weights=[0.4, 0.25, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05], 
    threshold=0.7,
    id_column='id'
)

In [ ]:
# st_corr_m2a.to_csv(PIPELINE_DIR / "st_corr_m2a.csv", index=False)
# st_corr_m2g.to_csv(PIPELINE_DIR / "st_corr_m2g.csv", index=False)

In [15]:
# data fusion for standard blocker
from PyDI.fusion import DataFusionStrategy, DataFusionEngine, longest_string, union, prefer_higher_trust
import pandas as pd

amazon_books.attrs["trust_score"] = 3
metabooks.attrs["trust_score"] = 2
goodreads.attrs["trust_score"] = 1

all_standard_correspondences = pd.concat([st_corr_m2a, st_corr_m2g], ignore_index=True)

len(all_standard_correspondences)

20134

In [20]:
strategy = DataFusionStrategy('books_fusion_strategy')

strategy.add_attribute_fuser('title_norm', longest_string)
strategy.add_attribute_fuser('author_norm', longest_string)
strategy.add_attribute_fuser('publish_year', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('publisher', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('isbn_clean', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('price', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('page_count', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('rating', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('genres', union)

In [21]:
engine = DataFusionEngine(strategy, debug=True, debug_format='json',
                          debug_file= PIPELINE_DIR / "debug_fusion_standard_blocker.jsonl")

fused_standard_blocker = engine.run(
    datasets=[amazon_books, metabooks, goodreads],
    correspondences=all_standard_correspondences,
    id_column="id",
    include_singletons=False,
)

print(f'Fused rows: {len(fused_standard_blocker):,}')
display(fused_standard_blocker.head())

Fused rows: 17,637


,_id,_fusion_group_id,_fusion_sources,title_norm,author,title,publisher,isbn_clean,publish_year,id,...,rating,rating_number,page_count,language,genres,price,_fusion_confidence,_fusion_metadata,book_format,edition
0,metabooks_104389,group_0,"[amazon_books, metabooks]",dawn of desire,Joyce Verrette,Dawn Of Desire,Avon Books,038000562X,1982.0,metabooks_104389,...,4.9,24,NaN,English,None,8.470000,0.615385,"{'title_norm_rule': 'longest_string', 'title_n...",NaN,NaN
1,goodreads_2244,group_1,"[goodreads, metabooks]",money,Martin Amis,Money,Vintage Books,0099461889,2005.0,goodreads_2244,...,3.8,22941,368.0,English,"[['Fiction', 'Classics', 'Contemporary', 'Nove...",10.960000,0.733333,"{'title_norm_rule': 'longest_string', 'title_n...",Paperback,nan
2,goodreads_32145,group_2,"[goodreads, metabooks]",classical electrodynamics,John David Jackson,Classical Electrodynamics,"Classical Electrodynamics, 3Rd Ed",9788126510948,1998.0,goodreads_32145,...,4.5,1373,832.0,English,"[['Engineering', 'Transportation'], ['Physics'...",63.209999,0.733333,"{'title_norm_rule': 'longest_string', 'title_n...",Hardcover,Third Edition
3,metabooks_269119,group_3,"[amazon_books, metabooks]",notorious the life of ingrid bergman,Donald Spoto,Notorious: The Life of Ingrid Bergman,HarperCollins Publishers,0060187026,1997.0,metabooks_269119,...,4.0,17,496.0,English,"[['Biographies', 'Memoirs', 'Arts', 'Literatur...",27.490000,0.769231,"{'title_norm_rule': 'longest_string', 'title_n...",NaN,NaN
4,amazon_6573,group_4,"[amazon_books, metabooks]",the med,David Poyer,The Med,St. Martin's Press,0312916450,1993.0,amazon_6573,...,3.9,544,NaN,English,None,36.950001,0.500000,"{'title_norm_rule': 'longest_string', 'title_n...",NaN,NaN


In [22]:
fused_standard_blocker.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17637 entries, 0 to 17636
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   _id                 17637 non-null  object 
 1   _fusion_group_id    17637 non-null  object 
 2   _fusion_sources     17637 non-null  object 
 3   title_norm          17637 non-null  object 
 4   author              17637 non-null  object 
 5   title               17637 non-null  object 
 6   publisher           17637 non-null  object 
 7   isbn_clean          17637 non-null  object 
 8   publish_year        17630 non-null  float64
 9   id                  17637 non-null  object 
 10  author_norm         17637 non-null  object 
 11  rating              17637 non-null  float64
 12  rating_number       17637 non-null  int64  
 13  page_count          15714 non-null  float64
 14  language            17543 non-null  object 
 15  genres              16766 non-null  object 
 16  pric

In [23]:
fused_standard_blocker.to_csv(PIPELINE_DIR / "fused_standard_blocker.csv")